# CareerCraft AI - Resume Role Classifier (ML Project)

**What this notebook does:** reads a resume and predicts which job category it belongs to (like HR, IT, Chef...).
It can also compare a resume with a job description and list the keywords the resume is missing.

**Before running:** upload `Resume.csv` (Kaggle Resume Dataset) using the folder icon on the left, then run the cells from top to bottom.

## Step 1: Import the libraries

In [ ]:
import pandas as pd
import re
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## Step 2: Load the data
The csv has many columns, we only need the resume text and its category (the correct answer).

In [ ]:
data = pd.read_csv("Resume.csv")
data = data[["Resume_str", "Category"]]
data = data.dropna()   # remove empty rows

print("total resumes:", len(data))
print(data["Category"].value_counts())

## Step 3: Clean the text
Computers treat "Python" and "python" as different words, so we make the text uniform:
lowercase, remove links/emails, remove numbers and symbols.

In [ ]:
def clean_text(text):
    text = text.lower()                            # all lowercase
    text = re.sub(r"http\S+|\S+@\S+", " ", text)   # remove links and emails
    text = re.sub(r"[^a-z+#\s]", " ", text)        # remove numbers and symbols (keep c++ and c#)
    text = re.sub(r"\s+", " ", text).strip()       # remove extra spaces
    return text

data["clean"] = data["Resume_str"].apply(clean_text)

print("BEFORE:", data["Resume_str"][0][:200])
print("AFTER:", data["clean"][0][:200])

## Step 4: Split into training data and testing data
80% of the resumes are used for learning, 20% are hidden and used later as an exam.
`stratify` keeps the same category proportions in both parts.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    data["clean"], data["Category"],
    test_size=0.2,
    random_state=42,
    stratify=data["Category"]
)

print("training resumes:", len(x_train))
print("testing resumes:", len(x_test))

## Step 5: Convert text to numbers (TF-IDF)
A model can only work with numbers, so every resume becomes a list of 20000 numbers.
A word gets a high number if it is common in that resume but rare in other resumes.
We learn the word list from the training data only (`fit_transform`) and reuse it for the test data (`transform`).

In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",    # ignore words like the, and, is
    ngram_range=(1, 2),      # use single words and pairs of words
    max_features=20000,      # keep the 20000 most useful words
    sublinear_tf=True
)

x_train_vec = tfidf.fit_transform(x_train)
x_test_vec = tfidf.transform(x_test)

print("training data shape:", x_train_vec.shape)

## Step 6: Train the model (Linear SVM)
For each of the 24 categories the model learns a weight for every word
(for example "payroll" gets a high weight for HR, "kitchen" gets a high weight for Chef).
When it sees a new resume it adds up the weights, and the category with the highest total is the prediction.

In [ ]:
model = LinearSVC()
model.fit(x_train_vec, y_train)
print("training done")

## Step 7: Test the model on the hidden 20%

In [ ]:
predictions = model.predict(x_test_vec)

print("Accuracy:", accuracy_score(y_test, predictions))
print()
print(classification_report(y_test, predictions))

A confusion matrix shows which categories get mixed up. The dark diagonal means correct predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 14))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, xticks_rotation=90, cmap="Blues", ax=ax, colorbar=False)
plt.title("Confusion Matrix - Linear SVM")
plt.show()

## Step 8: Predict the role for any resume
We clean the new text the same way, convert it with the same TF-IDF, then ask the model.

In [ ]:
def predict_role(resume_text):
    cleaned = clean_text(resume_text)
    vector = tfidf.transform([cleaned])
    answer = model.predict(vector)
    return answer[0]

## Step 9: Compare a resume with a job description
1. Predict the role of the resume and the role of the job description.
2. List the important job description words that are not in the resume.

In [ ]:
boring_words = ["looking", "required", "experience", "skills", "strong", "ability", "work", "team", "candidate"]

def compare(resume_text, jd_text):
    resume_role = predict_role(resume_text)
    jd_role = predict_role(jd_text)

    resume_vec = tfidf.transform([clean_text(resume_text)]).toarray()[0]
    jd_vec = tfidf.transform([clean_text(jd_text)]).toarray()[0]
    all_words = tfidf.get_feature_names_out()

    missing = []
    for i in jd_vec.argsort()[::-1][:60]:      # job description words, most important first
        word = all_words[i]
        if jd_vec[i] > 0 and resume_vec[i] == 0:
            skip = False
            for b in boring_words:
                if b in word:
                    skip = True
            if skip == False:
                missing.append(word)

    print("Resume looks like:", resume_role)
    print("Job description looks like:", jd_role)
    if resume_role == jd_role:
        print("Role match: YES")
    else:
        print("Role match: NO")
    print("Missing keywords:", missing[:10])

## Step 10: Try it with your own text (replace the text between the quotes)

In [ ]:
my_resume = """
Information technology graduate with experience in network administration, technical support,
troubleshooting hardware and software issues, SQL databases, Linux servers and Windows systems.
Managed user accounts, installed software and maintained the company network.
"""

my_jd = """
We are looking for an information technology professional with experience in cloud computing,
networking, system administration, SQL, cybersecurity and technical support.
"""

compare(my_resume, my_jd)

## Step 11: Save the trained model so it can be used later

In [ ]:
joblib.dump(model, "role_model.pkl")
joblib.dump(tfidf, "tfidf.pkl")
print("saved role_model.pkl and tfidf.pkl")